# Barriers: how vorflow meshes them, and the limits

A barrier is a line added with `add_line(..., is_barrier=True)`, for example a
fault. vorflow builds the grid so that the barrier is made of whole cell faces.
You can then put a horizontal flow barrier (HFB) on those faces in MODFLOW 6.
This notebook shows how the faces are placed, what vorflow reports about them,
and the cases it still does not handle.

**How the faces are placed.** vorflow cuts each barrier into segments of about
one `resolution`. It draws a circle around each segment end. The two circles of
a segment cross at two points, one on each side of the barrier. vorflow puts a
cell generator (the point that a Voronoi cell is built around) at each of these
points. If no other generator lies inside any of the circles, each segment is
exactly the face between its two generators. This is the empty-circle rule of
the UPR method (Berge et al., 2019). vorflow places these generators before
Gmsh runs and checks the rule. Gmsh then meshes the rest of the domain outside
the circles.

**What this means for the grid.** Each barrier segment is one straight cell
face. The line between the two cell centres crosses that face at a right
angle, so the orthogonality error on barrier faces is zero. No cell reaches
across a barrier, so no cell has to be cut. Two kinds of barrier are the
exception: a barrier with `quad_buffer=True` (section 7) and a barrier added
with `embed=False`, which only sets the mesh size. Both still go through the
old method described below.

**What changed.** vorflow 0.3 and earlier placed pairs of nodes on either side
of the barrier, and then cut any cell that still reached across it. A cut left
one of two defects:

- a **cut cell**: a cell with no mesh node of its own. Its `x`/`y` are its
  centroid, so the connections to its neighbours are not at right angles to
  their faces;
- a **face of several edges**: a small cut piece was merged into a neighbour,
  so two cells shared a bent face. MODFLOW 6 DISV uses only the first edge of a
  shared face to set the connection width, so it under-counts the flow area
  there.

Section 2 shows the cases that used to leave these defects. The plots still
look for both defects, so you can use the same check on your own grids.

Reference: Berge, R. L., Klemetsdal, Ø. S., and Lie, K.-A. (2019).
Unstructured Voronoi grids conforming to lower dimensional objects.
*Computational Geosciences* 23, 169–188.
<https://doi.org/10.1007/s10596-018-9790-0>

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from matplotlib.colors import Normalize
from shapely.geometry import LineString, box

import vorflow
from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, set_verbosity
from vorflow.utils import build_connectivity, calculate_mesh_quality

set_verbosity(0)

## Helpers

`build` meshes a 300 m square with some lines in it. Lines use `resolution=8`
unless a case sets their own. Keywords for `MeshGenerator` go in `mesher`.

`barrier_check` finds the two defects described above:

- a cut cell has a `node_id` above every mesh node and barrier mirror node,
  because cut pieces get new IDs;
- a face of several edges shows up as two neighbours whose shared boundary has
  more than one segment.

With exact barrier faces both counts should be 0. vorflow also warns when the
tessellator had to cut a cell on a barrier (see section 7), so this check is a
second, independent look.

`show` shades each cell by its **orthogonality error** from
`calculate_mesh_quality`. This is the largest angle, over the cell's faces,
between the line joining two cell centres and the normal to their shared face.
In a Voronoi grid it is 0 unless a cell was cut, so any shading marks a defect.
Each title gives the number of cells and the two defect counts. It also gives,
for cells within three cells of a barrier, the worst orthogonality error and
the lowest compactness. Compactness is the cell area relative to a circle of
the same perimeter. It is about 0.8 for a regular Voronoi cell and low for thin
cells.

`report` lists the entries of `MeshGenerator.barrier_report` as a table.

In [ ]:
DOMAIN = box(0, 0, 300, 300)
ORTHO = Normalize(0, 30)


def build(lines, mesher=None, **blueprint_kwargs):
    # Mesh DOMAIN with each (geometry, add_line keywords) in ``lines``.
    blueprint = ConceptualMesh(**blueprint_kwargs)
    blueprint.add_polygon(DOMAIN, zone_id=1, resolution=20)
    for k, (geometry, kwargs) in enumerate(lines):
        blueprint.add_line(geometry, line_id=f"line{k}", **{"resolution": 8, **kwargs})
    generator = MeshGenerator(background_lc=20, conceptual_mesh=blueprint, **(mesher or {}))
    generator.generate()
    tessellator = VoronoiTessellator(generator, blueprint)
    grid = tessellator.generate()
    return dict(blueprint=blueprint, mesher=generator, tessellator=tessellator, grid=grid)


def barrier_check(model):
    # Boolean mask of cut cells, and the shared boundaries made of several edges.
    grid = model["grid"]
    last_node = int(model["mesher"].node_tags.max()) + model["tessellator"].n_barrier_mirrors
    cut = grid["node_id"].to_numpy() > last_node
    connections = build_connectivity(grid, center="generator")
    several_edges = []
    for a, b in zip(connections["cell_id_1"], connections["cell_id_2"]):
        shared = grid.geometry.iloc[a].boundary.intersection(grid.geometry.iloc[b].boundary)
        parts = [g for g in getattr(shared, "geoms", [shared]) if g.geom_type == "LineString"]
        merged = shapely.line_merge(shapely.MultiLineString(parts))
        if sum(len(g.coords) - 1 for g in getattr(merged, "geoms", [merged])) > 1:
            several_edges.append(merged)
    return cut, several_edges


def near_barrier(model):
    # Cells within three barrier cells of a barrier.
    barriers = model["blueprint"].clean_lines.query("is_barrier")
    zone = shapely.union_all([row.geometry.buffer(3 * row["lc"]) for _, row in barriers.iterrows()])
    return model["grid"].intersects(zone).to_numpy()


def show(ax, model, window, title, circles=False):
    # Plot the grid inside ``window``, shaded by orthogonality error, with barrier defects marked.
    grid = model["grid"]
    connections = build_connectivity(grid, center="generator")
    quality = calculate_mesh_quality(grid, calc_ortho=True, connectivity=connections)
    cut, several_edges = barrier_check(model)
    near = near_barrier(model)
    quality.plot(ax=ax, column="ortho_error", cmap="Reds", norm=ORTHO, edgecolor="0.6", lw=0.5)
    if cut.any():
        grid[cut].plot(ax=ax, facecolor="none", edgecolor="k", hatch="///", lw=0.8)
    for face in several_edges:
        for part in getattr(face, "geoms", [face]):
            ax.plot(*part.xy, color="#00a6d6", lw=4, solid_capstyle="round")
    if circles and model["mesher"].barrier_circles is not None:
        centres, radii = model["mesher"].barrier_circles
        for (x, y), r in zip(centres, radii):
            ax.add_patch(plt.Circle((x, y), r, fill=False, color="tab:blue", lw=0.8))
        sites = model["mesher"].nodes[model["mesher"].node_is_site]
        ax.plot(sites[:, 0], sites[:, 1], "o", color="tab:blue", ms=3)
    faces = model["mesher"].barrier_lines
    solid = [] if faces is None else list(faces.geometry)
    for _, row in model["blueprint"].clean_lines.iterrows():
        if not row["is_barrier"]:
            ax.plot(*row.geometry.xy, color="k", lw=1.5, ls="--")
            continue
        ax.plot(*row.geometry.xy, color="k", lw=1.5, ls=":")
        if row["quad_buffer"]:
            solid.append(row.geometry)
    for line in solid:
        for part in getattr(line, "geoms", [line]):
            ax.plot(*part.xy, color="k", lw=2.5)
    ax.plot(grid["x"][~cut], grid["y"][~cut], "k.", ms=2)
    ax.set_xlim(window[0], window[2])
    ax.set_ylim(window[1], window[3])
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(
        f"{title}\n{len(grid)} cells, {cut.sum()} cut, {len(several_edges)} faces of several edges\n"
        f"near barrier: orthogonality error ≤ {quality['ortho_error'][near].max():.0f}°, "
        f"compactness ≥ {quality['compactness'][near].min():.2f}",
        fontsize=9,
    )


def compare(panels, ncols=None, circles=False):
    # Plot (title, model, window) panels side by side, ``ncols`` per row.
    ncols = ncols or len(panels)
    nrows = -(-len(panels) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols + 1, 6 * nrows), squeeze=False)
    for ax, (title, model, window) in zip(axes.flat, panels):
        show(ax, model, window, title, circles=circles)
    for ax in axes.flat[len(panels):]:
        ax.set_visible(False)
    fig.colorbar(plt.cm.ScalarMappable(norm=ORTHO, cmap="Reds"), ax=axes, shrink=0.7,
                 label="orthogonality error (degrees; 0 ideal)", extend="max")
    plt.show()


def report(model, *keys):
    # The barrier_report entries under ``keys`` as one table.
    entries = model["mesher"].barrier_report
    rows = [dict(entry=key, **row) for key in keys for row in entries[key]]
    return pd.DataFrame(rows)

In the plots, a solid black line is a barrier made of cell faces. A dotted
black line is a part of a barrier without faces, such as a spike tip
(section 5). A dashed black line is a standard (non-barrier) line. Small black
dots are cell generators. **Red** shading marks connections that are not at
right angles to their faces. **Hatched** cells are cut cells, and **cyan**
lines are faces of several edges. None of these three should appear, except
in section 7.

## 1. How the faces are placed

The barrier below has two corners. The blue circles are the empty circles, and
the blue dots are the generators placed where neighbouring circles cross. Each
barrier segment between two circle centres is the face between the two blue
generators on either side of it. Gmsh fills the space outside the circles with
ordinary mesh nodes.

At a corner, the first generator of each leg sits on the corner's circle. On
each side of the corner, the two legs then either share one generator or leave
a gap between their generators, depending on the angle. In a wedge narrower
than about 42 degrees, vorflow puts one row of cells across the wedge instead.

In [ ]:
fault = LineString([(30, 40), (140, 170), (160, 150), (270, 260)])
kink = build([(fault, dict(is_barrier=True))])
compare([("barrier with two corners, empty circles in blue", kink, (110, 130, 190, 200))], circles=True)

## 2. Cases that used to leave cut cells

In vorflow 0.3 each of the barriers below left cut cells or faces of several
edges at `resolution=8`. The old fix was a finer `resolution`, which did not
always work. All of them now mesh without either defect at the same
resolution, with no other settings:

- a **short leg**: an 8 m leg between two corners, one cell long;
- a **standard (non-barrier) line crossing the barrier next to a corner**;
- a **finely drawn curve**: a curve digitised every 4 m, half the resolution.
  Each digitised segment becomes its own face, so the faces follow the curve
  as drawn;
- a **T-junction** and a **crossing** of two barriers;
- a barrier that **ends on the domain boundary** at an angle.

In [ ]:
jog = LineString([(30, 40), (140, 170), (145.66, 164.34), (270, 260)])
crossing = LineString([(100, 250), (180, 100)])
xs = np.linspace(40, 260, 61)
curve = LineString(np.column_stack([xs, 150 + 40 * np.sin((xs - 40) / 220 * 2 * np.pi)]))
through = LineString([(40, 150), (260, 150)])
stem = LineString([(150, 150), (150, 270)])
across = LineString([(150, 30), (150, 270)])
to_boundary = LineString([(150, 180), (300, 120)])

barrier = dict(is_barrier=True)
compare([
    ("8 m leg between two corners", build([(jog, barrier)]), (115, 140, 175, 195)),
    ("standard line crossing next to a corner", build([(fault, barrier), (crossing, {})]), (115, 135, 185, 205)),
    ("curve digitised every 4 m", build([(curve, barrier)]), (60, 120, 160, 200)),
    ("T-junction", build([(through, barrier), (stem, barrier)]), (120, 120, 180, 180)),
    ("crossing", build([(through, barrier), (across, barrier)]), (120, 120, 180, 180)),
    ("barrier ending on the boundary", build([(to_boundary, barrier)]), (240, 100, 300, 160)),
], ncols=3)

The faces of the curve follow its digitised vertices, so they are as short as
the vertex spacing. If the vertices are only digitising detail,
`simplify_tolerance` on `add_line` removes them (section 4). It moves the line
by up to the tolerance. That is a change to the conceptual model, so decide it
on hydrogeological grounds, not to tidy the grid.

## 3. Barriers close to each other or to the boundary

Two nearly parallel barriers closer than about 1.2 x `resolution` leave no
room for a cell on each side of each barrier. vorflow then puts **one row of
cells** between them. The generators of that row lie on the midline between
the two barriers, so each cell touches both barriers. A barrier closer than
about 0.6 x `resolution` to the domain boundary gets the same treatment. The
row of cells between the barrier and the boundary then has its generators on
the boundary.

Every narrow gap keeps its row, however narrow, so flow along the gap stays
connected and no barrier is moved. The same holds for the legs of one barrier
that run close together, such as a U-turn: the row of cells turns round the
bend.

The cells of the row are at most `resolution` long, and at most 3 x the gap.
A 3:1 cell has a compactness of about 0.59. So a narrow gap costs cells: about
`length / (3 x gap)` in the row, and the mesh around the row is graded down to
match. vorflow warns, with the count, when the cells of a row are shorter than
0.5 x `resolution`. `barrier_report["corridors"]` lists each row, with its kind
(`"barriers"` or `"boundary"`), the lines involved, its length, the gap, how
each end closes and its number of cells.

Below, two barriers 4 m apart, a barrier 3.2 m from the boundary, two barriers
1 m apart and a U-turn 1 m wide. The last two have many more cells than the
first, which is the price of keeping the gap open.

In [ ]:
parallel = build([
    (LineString([(40, 150), (260, 150)]), barrier),
    (LineString([(40, 154), (260, 154)]), barrier),
])
hugging = build([(LineString([(40, 3.2), (260, 3.2)]), barrier)])
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)  # the dense-row warning; its details are in the report
    close = build([
        (LineString([(40, 150), (260, 150)]), barrier),
        (LineString([(40, 151), (260, 151)]), barrier),
    ])
    u_turn = build([(LineString([(40, 150), (260, 150), (260, 151), (40, 151)]), barrier)])
compare([
    ("two barriers 4 m apart", parallel, (120, 130, 180, 175)),
    ("barrier 3.2 m from the boundary", hugging, (120, -5, 180, 40)),
    ("two barriers 1 m apart", close, (120, 135, 160, 165)),
    ("U-turn 1 m wide", u_turn, (230, 135, 270, 165)),
], ncols=2)
pd.concat([report(model, "corridors") for model in (parallel, hugging, close, u_turn)], ignore_index=True)

## 4. How barrier lines are cleaned, and the report

Digitised faults rarely meet exactly. An end may stop just short of another
fault, or run just past it. `ConceptualMesh.generate()` cleans every barrier
before meshing, in this order:

1. **Snap ends.** An end within `max(connectivity_tolerance, simplify_tolerance)`
   of another barrier, of a barrier vertex or of the domain boundary is moved
   onto it. An end that runs past another barrier by at most that distance is
   trimmed back to the crossing. An end is not snapped onto a barrier it runs
   alongside. A part of a barrier lying on the domain boundary is dropped, and
   the end left behind is moved onto the boundary.
2. **Join meeting points.** Junctions and crossings with other barriers and
   with standard lines become vertices shared by both lines. A point where a
   line crosses or touches itself becomes a vertex too.
3. **Simplify.** With `simplify_tolerance` above 0, the line is replaced by the
   fewest straight pieces between its own vertices that stay within the
   tolerance. Junctions, crossings and ends are kept. This step runs after the
   junctions are joined, so it cannot pull a splay off the fault it starts on.
4. **Remove digitising noise.** A vertex closer than 0.1 x `resolution` to its
   neighbour is dropped, even when `simplify_tolerance` is 0. vorflow warns
   with the largest move this causes.

Barrier lines are not densified. When `MeshGenerator.generate()` places the
faces, it drops any piece that still runs past a junction by less than
0.1 x `resolution`. We refer to such a piece as a **stub**.

`MeshGenerator.barrier_report` records each of these changes with its place, so
you can check that none of them changes the conceptual model in a way that
matters. The model below has three branches off one barrier:

- the left branch stops 0.5 m short of the barrier and has
  `simplify_tolerance=1`, so its end is snapped onto the barrier;
- the middle branch runs 0.5 m past the barrier with `simplify_tolerance=0`,
  so the 0.5 m stub is dropped;
- the right branch has a vertex 0.3 m from its neighbour, which is removed as
  digitising noise.

In [ ]:
main = LineString([(40, 150), (260, 150)])
near_miss = LineString([(80, 270), (80, 150.5)])
overshoot = LineString([(150, 270), (150, 149.5)])
noisy = LineString([(220, 270), (220, 210), (220.3, 209.9), (220, 150)])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    cleaned = build([
        (main, barrier),
        (near_miss, dict(is_barrier=True, simplify_tolerance=1.0)),
        (overshoot, barrier),
        (noisy, barrier),
    ])
for warning in caught:
    if "digitising noise" in str(warning.message):
        print(warning.message)
report(cleaned, "snaps", "stubs", "noise")

## 5. Sharp corners, zigzags and narrow junctions

A single corner, even a sharp spike, is made exact down to about 10 degrees.
A run of sharp corners with short legs, such as a zigzag, can need faces far
shorter than `resolution`. vorflow then changes only the bends it cannot make,
before Gmsh runs: the two legs end where they are 0.5 x `resolution` apart,
and a short chord joins them. With `barrier_tips="line"` (the default) one
barrier line runs on from the chord to the tip along the bisector, so the
barrier still reaches the tip and the line moves by at most 0.25 x
`resolution`. With `barrier_tips="cut"` the tip lies inside a few cells, so a
little flow can pass around it. Each changed bend is listed in
`barrier_report["relieved"]`, with one warning. The left plot keeps the tips as
lines; the right one cuts them.

`barrier_tips=None` keeps every bend as drawn: a spike sharper than
`sharp_angle` (18 degrees) then loses its tip (`barrier_report["tips"]`), and
any other bend that cannot be made exact raises `BarrierMeshError`. A bend with
a leg shorter than 0.5 x `resolution` is never changed: that is digitising
noise, and the error asks to resample the line.

In [ ]:
half = np.radians(12.5)
dx, dy = 32 * np.sin(half), 32 * np.cos(half)
zigzag = LineString([(150 - 3 * dx + i * dx, 150 + (dy / 2 if i % 2 else -dy / 2)) for i in range(7)])
line_tips = build([(zigzag, barrier)])
cut_tips = build([(zigzag, barrier)], mesher=dict(barrier_tips="cut"))
window = (150 - 3 * dx - 8, 150 - dy / 2 - 8, 150 + 3 * dx + 8, 150 + dy / 2 + 8)
compare([
    ('25° zigzag, barrier_tips="line" (default)', line_tips, window),
    ('25° zigzag, barrier_tips="cut"', cut_tips, window),
])
report(line_tips, "relieved")

**Narrow junctions.** Where two barrier legs meet at a junction at less than
`sharp_angle` (a splay, a T-branch or a crossing), or a barrier leaves the
domain boundary that narrowly, the sliver between them could only be filled
with ever smaller faces. vorflow merges the two legs over a short reach
instead. They share a short stem along their bisector and part at
1.5 x `sharp_angle` at its end. The stem is as long as it can be while no
point of a line moves more than 0.1 x `resolution`. A barrier leaving the
boundary slides along it instead, so it still ends on the boundary. Fans and
stars of legs closer than 29 degrees are merged pair by pair in the same way.
Each merge is listed in `barrier_report["collapsed"]`, with its angle, its
reach and the largest move, and the faces on a stem carry both barriers.

The left plot is a splay leaving a fault at 10 degrees, where the fault has a
digitised vertex 1.5 m past the junction, bent 0.8 m towards the splay. The
splay crosses the fault again just past that vertex. Before these merges this
model raised `BarrierMeshError` at `simplify_tolerance=0`. The right plot is a
barrier leaving the bottom boundary at 5 degrees.

In [ ]:
junction = (100, 150)
main_fault = LineString([(20, 150), junction, (101.52, 150.8), (280, 150)])
splay = LineString([junction, (100 + 170 * np.cos(np.radians(10)), 150 + 170 * np.sin(np.radians(10)))])
shallow = LineString([(40, 0), (40 + 300 * np.cos(np.radians(5)), 300 * np.sin(np.radians(5)))])
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)  # the merge warning; its details are in the report
    splayed = build([(main_fault, barrier), (splay, barrier)])
    leaving = build([(shallow, barrier)])
compare([
    ("10° splay past a bent fault vertex", splayed, (90, 135, 140, 175)),
    ("barrier leaving the boundary at 5°", leaving, (30, -5, 110, 30)),
])
pd.concat([report(splayed, "collapsed"), report(leaving, "collapsed")], ignore_index=True)

## 6. When vorflow cannot place the faces

Some inputs would need barrier faces far shorter than the resolution, for
example where two barrier parts nearly touch. vorflow then raises
`vorflow.BarrierMeshError` (a `ValueError`) instead of returning a damaged
grid. The message names the place, what makes the faces that short there, and
what helps for that cause.

**A near miss with no tolerance.** The branch below stops 0.1 m short of the
other barrier. With `simplify_tolerance=0` and the default
`connectivity_tolerance`, the end is too far away to be snapped. The faces
between the end and the other barrier would have to be about 0.1 m long, so
vorflow raises, and the message says that a `simplify_tolerance` of at least
the gap snaps the end. With `simplify_tolerance=1` the end is snapped onto the
barrier and the model meshes.

In [ ]:
branch = LineString([(150, 270), (150, 150.1)])
try:
    build([(through, barrier), (branch, barrier)])
except vorflow.BarrierMeshError as error:
    print(error)
snapped = build([(through, barrier), (branch, dict(is_barrier=True, simplify_tolerance=1.0))])

**Two barriers ending together at 1 degree.** Two lines that end at the same
point form a narrow V. The spike rule leaves the last part of the tip without
faces, but past it the two legs stay a fraction of `resolution` apart for a
long way, so vorflow raises. The message names the narrow angle and points to
`quad_buffer=True`, which meshes such a barrier with a structured strip of
cells instead of exact faces (section 7).

In [ ]:
tip = (150, 150)
v_legs = [LineString([tip, (270, 150)]), LineString([tip, (150 + 120 * np.cos(np.radians(1)), 150 + 120 * np.sin(np.radians(1)))])]
try:
    build([(v_legs[0], barrier), (v_legs[1], barrier)])
except vorflow.BarrierMeshError as error:
    print(error)
compare([("near miss, simplify_tolerance=1", snapped, (125, 130, 175, 175))])

## 7. Quad buffers on barriers (still cut)

`quad_buffer=True` builds a strip of structured quads along a line. On a
barrier, the strip does not use the exact faces described above. It still uses
the old method: the tessellator mirrors generators across the barrier and cuts
the cells that still reach across it. So cut cells and faces of several edges
can still appear:

- on a barrier with corners, the rows of quad nodes on the two sides drift
  apart after the first corner, and cells along the rest of the line are cut;
- `quad_buffer_thickness=2` puts a row of nodes on the barrier itself, so every
  cell along it is cut.

Unless you need the structured strip, leave `quad_buffer` off on barriers. The
right-hand plots show the same barriers with exact faces.

In [ ]:
straight = LineString([(40, 40), (260, 200)])
compare([
    ("quad buffer, corners", build([(fault, dict(is_barrier=True, quad_buffer=True))]), (110, 120, 200, 210)),
    ("exact faces, corners", kink, (110, 120, 200, 210)),
])
compare([
    ("quad_buffer_thickness=2", build([(straight, dict(is_barrier=True, quad_buffer=True,
                                                       quad_buffer_thickness=2))]), (110, 70, 190, 150)),
    ("exact faces", build([(straight, barrier)]), (110, 70, 190, 150)),
])

## Summary

| Case | What vorflow does | What to do |
|---|---|---|
| Corners, short legs, curves, junctions, crossings, boundary ends | Exact faces | Nothing |
| Nearly parallel barriers closer than about 1.2 x `resolution`, legs of one barrier as close (a U-turn), or a barrier close to the boundary | One shared row of cells, at most 3 x the gap long (`barrier_report["corridors"]`); a warning with the cell count when they are shorter than 0.5 x `resolution` | If the row costs too many cells, and the model allows it, move the barriers further apart |
| Junctions or boundary exits narrower than `sharp_angle`, fans narrower than 29 degrees | Merged over a short reach, moving the lines at most 0.1 x `resolution`, with a warning (`barrier_report["collapsed"]`) | Check the moves; the faces on a stem carry both barriers |
| Ends that nearly meet another barrier | Snapped within `max(connectivity_tolerance, simplify_tolerance)` (`barrier_report["snaps"]`) | Check the snaps |
| Vertices closer than 0.1 x `resolution` | Dropped, with a warning (`barrier_report["noise"]`) | Check the largest move |
| Pieces past a junction shorter than 0.1 x `resolution` | Dropped (`barrier_report["stubs"]`) | Check the stubs |
| A bend of a spike or zigzag that cannot be made exact | Only that bend is changed: the legs end 0.5 x `resolution` apart, and one line runs on to the tip (`barrier_tips="line"`) or the tip lies inside cells (`"cut"`), with a warning (`barrier_report["relieved"]`) | Check the moves, or use a finer `resolution` there to keep the bend exact |
| Faces would have to be far shorter than `resolution` | `BarrierMeshError` naming the place and the cause | Follow the message: the `simplify_tolerance` it gives for near misses and short folds, resampling at the resolution for digitising noise (`vorflow.utils.resample_geometry`), the finer `resolution` it gives, `quad_buffer=True` for narrow folds of long legs with `barrier_tips=None` |
| A point feature on a barrier or at its end | `BarrierMeshError` naming the point | Move the point off the line, or end the barrier short of it |
| `quad_buffer=True` or `embed=False` on a barrier | Old method: mirrored generators and cut cells | Leave `quad_buffer` off on barriers |

Read `barrier_report` for any model with barriers before you build HFB input.
It lists every place where vorflow changed a barrier or left part of it
without faces.